# Ontologies

`seqout` can link words to biomedical concepts using its knowlegde graph built from  MONDO, MeSH, HGNC,
ChEBI, Uberon, Cell Ontology (CL), EFO and Cellosaurus (CVCL).

This notebook shows how to:

1. look up a term in the graph,
2. control how search uses synonyms,
3. find samples by ontology ID,
4. add ontology IDs to your own labels.

In [1]:
import pandas as pd

from seqout import connect

sq = connect()

## Look up a term

`sq.ontology()` returns the term, its IDs, its synonyms and its children.

In [2]:
liver = sq.ontology("liver")

print("IDs:      ", liver.xrefs)
print("sources:  ", liver.sources)
print("synonyms: ", [s.name for s in liver.synonyms])
print("children: ", len(liver.children))

IDs:       ['UBERON:0002107', 'MeSH:D008099']
sources:   ['MeSH', 'UBERON']
synonyms:  ['iecur', 'jecur', 'livers']
children:  35


IDs are CURIEs (`SOURCE:ID`), one per ontology that knows the term. Children
are narrower terms, such as parts of the liver.

In [3]:
for child in liver.children[:5]:
    print(f"{child.name:35} {', '.join(child.xrefs)}")

bile ducts, intrahepatic            MeSH:D001653
biliary ductule                     UBERON:0004058
capsule of liver                    UBERON:0016479
central vein of liver               UBERON:0006841
fetal liver hematopoietic progenitor cell CL:0002353


Synonyms show why a search for one word finds studies that use another:

In [4]:
masld = sq.ontology("masld", children=False)

for s in masld.synonyms[:5]:
    print(f"{s.name:60} {', '.join(s.xrefs)}")

metabolic dysfunction associated steatotic liver disease     MONDO:0013209
non alcoholic fatty liver                                    EFO:1001248, MONDO:0013209
non alcoholic fatty liver disease                            MeSH:D065626, MONDO:0013209
nonalcoholic fatty liver disease                             MeSH:D065626, MONDO:0013209
nafld nonalcoholic fatty liver disease                       MONDO:0013209


Useful options:

| Option | What it does |
| --- | --- |
| `max_hops=1..4` | How far to follow synonym links (default 2). More hops, more synonyms. |
| `children=False` | Skip the children. Much faster when you only need IDs. |

A term that is not in the graph returns `None`:

In [5]:
print(sq.ontology("zzznope"))

None


## Synonyms in search

By default, `search` also matches synonyms. Compare the number of GEO studies
for "masld" with and without them:

In [6]:
with_synonyms = sq.search_facets("masld", db="geo").total
exact = sq.search_facets("masld", db="geo", structured=True).total
without_mondo = sq.search_facets("masld", db="geo", exclude_ontology=["MONDO"]).total

pd.Series(
    {"with synonyms": with_synonyms, "exact words only": exact, "without MONDO": without_mondo},
    name="GEO studies",
)

with synonyms       1053
exact words only     266
without MONDO        266
Name: GEO studies, dtype: int64

To turn synonyms off or drop one source in `search`, use:

```python
sq.search("masld", expand=False)                  # exact words only
sq.search("masld", exclude_ontology=["MeSH"])      # all sources except MeSH
```

Here, all extra matches come from MONDO, so removing it gives the same count as
an exact search.

## Find samples by ontology ID

The `sample_search` method takes an ID and finds relevant samples.

In [7]:
cohort = sq.sample_search(disease_ontology_id="MONDO:0013209", limit=5)
print(cohort.total, "samples, including narrower diseases")

exact_only = sq.sample_search(
    disease_ontology_id="MONDO:0013209", include_descendants=False, limit=1
)
print(exact_only.total, "samples with exactly this ID")

cohort.to_df()[["sample", "study_accession", "disease", "disease_ontology_id", "tissue"]]

14632 samples, including narrower diseases


9192 samples with exactly this ID


,sample,study_accession,disease,disease_ontology_id,tissue
0,Control_Rep1,E-MTAB-10598,Non-Alcoholic Fatty Liver Disease,MONDO:0013209,Liver
1,Control_Rep2,E-MTAB-10598,Non-Alcoholic Fatty Liver Disease,MONDO:0013209,Liver
2,Control_Rep3,E-MTAB-10598,Non-Alcoholic Fatty Liver Disease,MONDO:0013209,Liver
3,DEN Heterozygous Tid1,E-MTAB-9974,Non-Alcoholic Steatohepatitis (Nash),MONDO:0007027,Liver
4,DEN Tid1 KO,E-MTAB-9974,Non-Alcoholic Steatohepatitis (Nash),MONDO:0007027,Liver


## Add ontology IDs to your own labels

Submitters use free text for cell types and tissues. `map_to_ontology` adds a
`<column>_ontology_id` column next to each column you name. Labels it does not
know become `NaN`.

In [8]:
cells = pd.DataFrame(
    {"celltype": ["T cell", "B cell", "monocyte", "NK cell", "naive B cell", "unknown"]}
)
sq.map_to_ontology(cells, "celltype")

,celltype,celltype_ontology_id
0,T cell,"CL:0000084,MeSH:D013601"
1,B cell,CL:0000236
2,monocyte,"CL:0000576,MeSH:D009000"
3,NK cell,"CL:0000623,MeSH:D007694"
4,naive B cell,CL:0000788
5,unknown,NaN


To keep IDs from only one ontology, pass `ontology=`. For cell types, the Cell
Ontology is the usual choice:

In [9]:
sq.map_to_ontology(cells, "celltype", ontology="CL")

,celltype,celltype_ontology_id
0,T cell,CL:0000084
1,B cell,CL:0000236
2,monocyte,CL:0000576
3,NK cell,CL:0000623
4,naive B cell,CL:0000788
5,unknown,NaN


Useful options:

| Option | What it does |
| --- | --- |
| `ontology="CL"` | Keep IDs from one source only. |
| `use_synonyms=True` | If a label has no ID, borrow one from a synonym. Can be less precise. |
| `max_hops=1..4` | How far to follow synonyms (default 1). |

It sends one request per distinct label, so a column with a few unique values is
fast even on millions of cells. To label an AnnData, pass `adata.obs`.

In [10]:
sq.close()

The command line has the same lookup as `seqout onto`; see
[Command line](../cli.md#ontology).